# 04 · Evaluation and Error Analysis

Loads every experiment's saved predictions (`results/<experiment>/`) and produces the comparison tables, plots, error analysis and qualitative examples used in the report and demo video.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os
os.environ.setdefault("AFRIMED_ROOT", str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
from afrimed import plots
from afrimed.plots import load_all_metrics, results_table
from afrimed.config import RESULTS_DIR
df = load_all_metrics()
table = results_table(df)
table.round(4)

In [ ]:
table.round(4).to_csv(RESULTS_DIR / "summary.csv", index=False)
BEST = df[df.group != "baseline"].sort_values("mcq_accuracy", ascending=False).name.iloc[0]
print("best experiment by MCQ accuracy:", BEST)

## 1. Experiment comparison

In [ ]:
plots.plot_experiment_comparison(df); plt.show()

In [ ]:
trained = [n for n in df.name if (RESULTS_DIR / n / "train_log.json").exists()]
plots.plot_loss_curves(trained); plt.show()

In [ ]:
plots.plot_lora_ablation(df); plt.show()

In [ ]:
plots.plot_efficiency(df); plt.show()

## 2. Fine-grained MCQ analysis

In [ ]:
compare = [n for n in ["baseline_tfidf", "zs_qwen05", "ft_flant5_base", "lora_qwen05_r16", BEST] if n in set(df.name)]
compare = list(dict.fromkeys(compare))
plots.plot_specialty_heatmap(compare); plt.show()

In [ ]:
for n in [m for m in ["zs_qwen05", BEST] if m in set(df.name)]:
    plots.plot_confusion(n); plt.show()

In [ ]:
plots.plot_calibration([m for m in ["zs_qwen05", "lora_qwen05_r16", BEST] if m in set(df.name)]); plt.show()
df[["name", "mcq_accuracy", "mcq_ece"]].sort_values("mcq_ece")

In [ ]:
slices = plots.plot_accuracy_slices([m for m in ["zs_qwen05", BEST] if m in set(df.name)]); plt.show()
slices.pivot_table(index=["slice", "value"], columns="model", values="accuracy").round(3)

## 3. Short-answer (generative) analysis

In [ ]:
gen_models = [n for n in ["baseline_tfidf", "zs_qwen05", "ft_flant5_base", "lora_qwen05_r16", "lora_qwen05_r16_rag", BEST] if n in set(df.name)]
gen_models = list(dict.fromkeys(gen_models))
plots.plot_saq_distributions(gen_models); plt.show()

In [ ]:
plots.plot_length_scatter(BEST); plt.show()

## 4. Error analysis

In [ ]:
from afrimed.error_analysis import plot_error_profiles, export_examples
mcq_prof, saq_prof = plot_error_profiles(BEST); plt.show()
display(mcq_prof.round(3))
saq_prof.error_category.value_counts().to_frame("answers")

In [ ]:
examples = export_examples(BEST, n=5)
for title, t in examples.items():
    display(Markdown(f"### {title}"))
    display(t)

In [ ]:
for cat in ["Content mismatch (likely factual error)", "Over-generic / verbose", "Very short reference", "Truncated / rambling"]:
    sub = saq_prof[saq_prof.error_category == cat]
    if len(sub):
        display(Markdown(f"**{cat}** ({len(sub)} answers)"))
        display(sub[["question", "prediction", "rationale", "semantic_sim"]].head(3))

## 5. Behaviour on out-of-domain questions (without the domain gate)

In [ ]:
probe = []
for n in [m for m in ["zs_qwen05", BEST] if (RESULTS_DIR / m / "ood_probe_responses.csv").exists()]:
    probe.append(pd.read_csv(RESULTS_DIR / n / "ood_probe_responses.csv").assign(model=n))
probe = pd.concat(probe) if probe else pd.DataFrame()
probe.pivot_table(index="question", columns="model", values="response", aggfunc="first").head(10) if len(probe) else None

Without the gate, a fine-tuned model still tries to answer non-medical questions (often turning them into medical ones). This motivates the explicit **domain gate** from notebook 02, which the web app applies *before* calling the language model.

In [ ]:
plots.plot_ood(); plt.show()